# Practice 16 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — Basics

### Task 1: DateTime + MultiIndex
A coffee shop sales DataFrame has been created for you.

1. Convert `date` to datetime, then extract `year` and `month` as new columns
2. Add a `days_ago` column = days between `date` and `2026-03-01`
3. Find mean and median `days_ago` using NumPy
4. Set a MultiIndex from `city` and `product`, sort it
5. Retrieve all rows for `'Berlin'`
6. Use `pd.IndexSlice` to get all `'Latte'` rows across every city

In [7]:
# Starter data — don't change this
sales = pd.DataFrame({
    'city':     ['Berlin', 'Berlin', 'Paris', 'Paris', 'Tokyo', 'Tokyo', 'Berlin', 'Tokyo'],
    'product':  ['Latte', 'Espresso', 'Latte', 'Espresso', 'Latte', 'Espresso', 'Cappuccino', 'Cappuccino'],
    'date':     ['2024-01-15', '2024-03-22', '2024-06-01', '2024-08-10',
                 '2024-11-05', '2025-01-20', '2025-04-07', '2025-07-14'],
    'revenue':  [4200, 3100, 5800, 2900, 6700, 3400, 4900, 5100],
    'units':    [210, 155, 290, 145, 335, 170, 245, 255],
})

# Your code here
sales['date'] = pd.to_datetime(sales['date'])
sales['year'] = sales['date'].dt.year
sales['month'] = sales['date'].dt.month
sales['days_ago'] = (pd.to_datetime('2026-03-01')-sales['date']).dt.days
np.mean(sales['days_ago'])
np.median(sales['days_ago'])
sales = sales.set_index(['city','product']).sort_index()
sales.loc['Berlin']
idx = pd.IndexSlice
sales.loc[idx[:,'Latte'],:]

,,date,revenue,units,year,month,days_ago
city,product,,,,,,
Berlin,Latte,2024-01-15,4200,210,2024,1,776
Paris,Latte,2024-06-01,5800,290,2024,6,638
Tokyo,Latte,2024-11-05,6700,335,2024,11,481


---
## Level 2 — Transformations

### Task 2: stack() / unstack() + Custom Logic
A wide DataFrame of city pollution readings (µg/m³) has been created for you
(cities as the index, months as columns).

1. Stack into long format — each row one `(city, month)` pair. Store as `pollution_long`.
2. Add a `log_reading` column using NumPy
3. Add a `high_pollution` column: `True` if reading > 50 (use `np.where`)
4. Unstack to restore wide format
5. Find the city with highest mean pollution using NumPy

In [19]:
# Starter data — don't change this
np.random.seed(5)
pollution = pd.DataFrame({
    'city': ['Cairo', 'Delhi', 'Lagos', 'Seoul'],
    'Jan':  np.random.randint(30, 90, size=4),
    'Apr':  np.random.randint(30, 90, size=4),
    'Jul':  np.random.randint(30, 90, size=4),
    'Oct':  np.random.randint(30, 90, size=4),
}).set_index('city')

# Your code here
pollution_long = pollution.stack()
pollution_long = pollution_long.to_frame('reading')
pollution_long['log_reading'] = np.log(pollution_long['reading'])
pollution_long
pollution_long['high'] = np.where(pollution_long['reading']>50, True, False)
pw = pollution_long.unstack()
pollution_long
cm = {}
for city in pollution_long.index.get_level_values('city').unique():
    cm[city]=np.mean(pollution_long.loc[city,'reading'])
max(cm,key=cm.get)



'Cairo'

### Task 3: .melt() ✨ New

`.melt()` unpivots a wide DataFrame to long format — columns become rows. It's similar to `.stack()`, but works on regular columns rather than a MultiIndex.

```python
pd.melt(df, id_vars=['id_col'], value_vars=['col1', 'col2'],
        var_name='variable', value_name='value')
```

- `id_vars` — columns to keep as-is (they repeat for each melted row)
- `value_vars` — columns to melt into rows (their names go into `var_name`, their values into `value_name`)
- If `value_vars` is omitted, all non-`id_vars` columns are melted

Example:
```
   region   Q1    Q2
   North   100   150
```
After `pd.melt(df, id_vars=['region'], var_name='quarter', value_name='sales')`:
```
   region  quarter  sales
   North   Q1       100
   North   Q2       150
```

A wide quarterly sales DataFrame has been created for you (one row per region, one column per quarter).

1. Use `.melt()` to reshape it to long format — one row per `(region, quarter)`. Use `var_name='quarter'` and `value_name='sales'`. Store as `sales_long`.
2. Find mean sales per quarter using groupby
3. Find the `(region, quarter)` combination with the highest sales — use NumPy

In [79]:
df = pd.DataFrame({'a': [1,2,3], 'b': [4,5,6]})
sub = df[df['a'] > 1]
sub
sub['b']=99
df

/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_20947/1822375512.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub['b']=99


,a,b
0,1,4
1,2,5
2,3,6


In [37]:
# Starter data — don't change this
np.random.seed(99)
quarterly = pd.DataFrame({
    'region': ['North', 'South', 'East', 'West'],
    'Q1':     np.random.randint(20_000, 80_000, size=4),
    'Q2':     np.random.randint(20_000, 80_000, size=4),
    'Q3':     np.random.randint(20_000, 80_000, size=4),
    'Q4':     np.random.randint(20_000, 80_000, size=4),
})

# Your code here
sales_long = pd.melt(quarterly, id_vars='region',var_name='quarter',value_name='sales')
sales_long
sales_long.groupby('quarter')['sales'].mean()
max_index = np.argmax(sales_long['sales'])
max_region_quarter = sales_long.iloc[max_index]
max_region_quarter['region'] 
max_region_quarter['quarter']

'Q3'

### Task 4: groupby + transform + pd.cut
A staff performance DataFrame has been created for you.

1. Add a `team_avg_score` column = each row's team mean score (use `transform`)
2. Add an `above_team_avg` column: `True` if `score` exceeds `team_avg_score`
3. Bin `score` into 3 equal-width groups labelled `['Needs Improvement', 'Meets Expectations', 'Exceeds Expectations']`, store as `rating`
4. Count rows per `rating` (use `observed=True`)

In [42]:
# Starter data — don't change this
np.random.seed(21)
staff = pd.DataFrame({
    'name':  ['Ava', 'Ben', 'Cara', 'Dom', 'Elle', 'Finn', 'Gia', 'Hiro', 'Isla', 'Jake'],
    'team':  ['Alpha', 'Beta', 'Alpha', 'Beta', 'Gamma', 'Alpha', 'Gamma', 'Beta', 'Gamma', 'Alpha'],
    'score': np.random.randint(40, 100, size=10),
    'calls': np.random.randint(20, 200, size=10),
})

# Your code here
staff['team_avg_score']=staff.groupby('team')['score'].transform('mean')
staff['above_team_avg']=staff['score']>staff['team_avg_score']
staff['rating'] = pd.cut(staff['score'],3,labels=['Needs Improvement', 'Meets Expectations', 'Exceeds Expectations'])
staff.groupby('rating',observed=True).agg('count')

,name,team,score,calls,team_avg_score,above_team_avg
rating,,,,,,
Needs Improvement,3,3,3,3,3,3
Meets Expectations,2,2,2,2,2,2
Exceeds Expectations,5,5,5,5,5,5


---
## Level 3 — Aggregation

### Task 5: pd.merge() + .xs() + Rolling Windows
Two DataFrames have been created for you: `warehouses` and `shipments`.

1. Inner join `warehouses` and `shipments` on `warehouse_id`
2. Left join instead — how many extra rows appear, and why?
3. On the inner join result, set a `(warehouse_id, month)` MultiIndex and sort it
4. Use `.xs()` to pull out all rows for warehouse `'W02'`
5. On that slice, compute a 3-period rolling mean of `weight_kg`
6. Find mean `weight_kg` per `region` on the inner join result using groupby + NumPy

In [73]:
# Starter data — don't change this
np.random.seed(44)
warehouses = pd.DataFrame({
    'warehouse_id': ['W01', 'W02', 'W03', 'W04', 'W05'],
    'region':       ['North', 'South', 'North', 'East', 'East'],
    'capacity':     [5000, 3000, 4500, 6000, 2500],
})

shipments = pd.DataFrame({
    'shipment_id':  [f'S{i:02d}' for i in range(1, 13)],
    'warehouse_id': ['W01','W01','W01','W02','W02','W02','W03','W03','W04','W04','W06','W06'],
    'month':        ['Jan','Feb','Mar','Jan','Feb','Mar','Jan','Feb','Jan','Feb','Jan','Feb'],
    'weight_kg':    np.random.randint(100, 1000, size=12),
})

# Your code here
ij = pd.merge(warehouses,
              shipments,
              how='inner',
              on='warehouse_id')
lj = pd.merge(warehouses,
              shipments,
              how='left',
              on='warehouse_id')
print(ij.shape,lj.shape)
#because W05 is not in shipments
ij = ij.set_index(['warehouse_id','month']).sort_index()
slice = ij.xs('W02',level='warehouse_id')
sdf = slice.copy()
sdf['rollingmean'] = sdf['weight_kg'].rolling(3).mean()
ij.groupby('region')['weight_kg'].agg(np.mean)


(10, 6) (11, 6)


/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_20947/1121559079.py:31: FutureWarning: The provided callable <function mean at 0x7fbff745c430> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  ij.groupby('region')['weight_kg'].agg(np.mean)


region
East     925.000000
North    528.000000
South    647.666667
Name: weight_kg, dtype: float64

### Task 6: pd.concat() ✨ New

`pd.concat()` combines multiple DataFrames into one. Use it when you have the same kind of data split across multiple DataFrames (e.g. monthly batches, different files).

```python
pd.concat([df1, df2])                     # stack vertically (default) — adds rows
pd.concat([df1, df2], ignore_index=True)  # stack vertically and reset the index
pd.concat([df1, df2], axis=1)             # join side by side — adds columns
```

Two survey DataFrames from different months have been created for you. They have the same columns.

1. Concatenate `survey_jan` and `survey_feb` vertically. Reset the index. Store as `survey`.
2. How many rows does `survey` have?
3. Find mean `satisfaction` per `department` using groupby
4. Add a `high_satisfaction` column: `True` if `satisfaction` >= 4 (use `np.where`)
5. Find the department with the highest mean satisfaction — use NumPy

In [ ]:
# Starter data — don't change this
np.random.seed(7)
survey_jan = pd.DataFrame({
    'employee':     ['Ana', 'Ben', 'Cara', 'Dan', 'Eva'],
    'department':   ['Eng', 'Sales', 'Eng', 'HR', 'Sales'],
    'satisfaction': np.random.randint(1, 6, size=5),
    'month':        'Jan',
})

survey_feb = pd.DataFrame({
    'employee':     ['Finn', 'Gia', 'Hiro', 'Isla', 'Jake'],
    'department':   ['HR', 'Eng', 'Sales', 'HR', 'Eng'],
    'satisfaction': np.random.randint(1, 6, size=5),
    'month':        'Feb',
})

# Your code here
survey = pd.concat([survey_jan, survey_feb],ignore_index=True)
survey.shape
#10 rows
survey.groupby('department')['satisfaction'].mean()
survey['high_satisfaction'] = np.where(survey['satisfaction']>=4, True, False)
md = {}
for dept in survey['department'].unique():
    d = survey[survey['department']==dept]
    md[dept] = np.mean(d['satisfaction'])
max(md,key=md.get)


department
Eng      3.25
HR       3.00
Sales    3.00
Name: satisfaction, dtype: float64

---
## Level 4 — Real-world

### Task 7: Full Pipeline
Three DataFrames have been created for you: `athletes` (demographics), `results_2024` and `results_2025` (race times in seconds).

1. Concatenate `results_2024` and `results_2025` into `all_results` (reset the index)
2. Inner join `athletes` and `all_results` on `athlete_id`
3. Drop any rows with null values
4. Add a `pace_score` column = `(sprint * 0.5) + (endurance * 0.3) + (relay * 0.2)` — use `np.dot`
5. Bin `pace_score` into 3 equal-width groups labelled `['Bronze', 'Silver', 'Gold']`, store as `medal_tier`
6. Compute mean `pace_score` grouped by `country` and `year`, then `.unstack()` so countries are rows and years are columns
7. Find the correlation between `sprint` and `endurance` times using NumPy

In [ ]:
# Starter data — don't change this
np.random.seed(55)
athletes = pd.DataFrame({
    'athlete_id': [f'A{i:03d}' for i in range(1, 16)],
    'name':       [f'Athlete_{i}' for i in range(1, 16)],
    'country':    np.random.choice(['USA', 'GBR', 'JPN', 'BRA'], size=15),
})

# results_2025 has 13 rows — 2 athletes missing
results_2024 = pd.DataFrame({
    'athlete_id': [f'A{i:03d}' for i in range(1, 16)],
    'sprint':     np.random.randint(10, 30, size=15),
    'endurance':  np.random.randint(120, 300, size=15),
    'relay':      np.random.randint(40, 80, size=15),
    'year':       2024,
})

results_2025 = pd.DataFrame({
    'athlete_id': [f'A{i:03d}' for i in range(1, 14)],
    'sprint':     np.random.randint(10, 30, size=13),
    'endurance':  np.random.randint(120, 300, size=13),
    'relay':      np.random.randint(40, 80, size=13),
    'year':       2025,
})

# Your code here

all_results = pd.concat(
    [results_2024, results_2025],
    ignore_index=True
)
ij = pd.merge(
    athletes,
    all_results, 
    how='inner',
    on='athlete_id'
)
ij = ij.dropna()
ij['pace_score'] = np.dot(ij[['sprint','endurance','relay']],[0.5,0.3,0.2])
ij['medal_tier'] = pd.cut(ij['pace_score'],3,labels=['Bronze','Silver','Gold'])
ij.groupby(['country','year'])['pace_score'].mean().unstack()
np.corrcoef(ij['sprint'],ij['endurance'])

array([[1.        , 0.25408045],
       [0.25408045, 1.        ]])

In [85]:
ij['weight_kg'].memory_usage()


551

In [86]:
ij['weight_kg'].astype('int32').memory_usage()

511